# 03 — Identifiability: what one trajectory can and cannot reveal

Phase 1 established that the single-trajectory exponent estimator does not merely have high variance — for CTRW and scaled Brownian motion it *converges to the wrong answer* (both march toward $\alpha \sim 1$ regardless of their true exponent).

So the Phase 2 question is: **Can a single trajectory reveal which mechanism generated it?**

If it can, the exponent becomes recoverable: identify the mechanism, then apply the appropriate physical correction. If it cannot, that defines an **information wall**.

Read `PROJECT.md` and `PHASE2_PROMPT.md` first. All logic lives in `collectivediff` — this notebook imports, calls, and plots.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

from collectivediff.features import (
    FEATURE_GROUPS, FEATURE_NAMES, extract_feature_dict, extract_features,
)
from collectivediff.inference import (
    DatasetConfig, load_or_generate_dataset,
)
from collectivediff.studies_phase2 import (
    PHASE2_RESULTS_ROOT,
    run_step2_dataset_study,
    run_step3_classification_study,
    run_step4_ablation_study,
    run_step5_information_floor_study,
    run_step6_conditional_exponent_study,
)
from collectivediff.viz import (
    plot_conditional_exponent_comparison,
    plot_confusion_matrix_family,
    plot_feature_importance_ranking,
    plot_information_floor_efficiency,
    plot_pairwise_error_vs_length,
)

## 1. Scale-Invariant Feature Extraction & Dataset

The **isolated observer** sees a single trajectory $x(t)$ of shape `(n_steps, n_dim)` and nothing else. Every feature must be **invariant under spatial rescaling** $x \to c \cdot x$ ($c > 0$), so the classifier learns the generative mechanism rather than the diffusion coefficient $D$.

We generate datasets across continuous parameter sweeps for the 6 mechanisms (Brownian, fBm, CTRW, SBM, DDM, Lévy walk) across observation windows $T \in \{128, 512, 2048, 8192\}$.

In [ ]:
dataset_summary = run_step2_dataset_study(lengths=(128, 512, 2048, 8192), n_samples_per_class=120)
print("Feature extraction timing and shapes:")
for length in dataset_summary["trajectory_lengths"]:
    t_sec = dataset_summary["timings_seconds"][str(length)]
    shape = dataset_summary["dataset_shapes"][str(length)]
    print(f"  T = {length:5d}: {shape[0]} trajectories, {shape[1]} features extracted in {t_sec:.3f} s")

## 2. Classification as a Function of Trajectory Length $T$

We fit interpretable models (Multinomial Logistic Regression, Linear Discriminant Analysis) and Gradient Boosting as an upper bound, producing a family of confusion matrices indexed by $T$.

In [ ]:
clf_summary = run_step3_classification_study(lengths=(128, 512, 2048, 8192), n_samples_per_class=120)

# Display family of confusion matrices across T for Gradient Boosting
fig_cm = plot_confusion_matrix_family(
    results_by_length=clf_summary["results_by_length"],
    classifier_name="gradient_boosting",
    lengths=(128, 512, 2048, 8192),
    mechanism_names=clf_summary["mechanism_names"],
)
plt.show()

## 3. Information Walls & Pairwise Separability

Pairs whose misclassification error rate plateaus above chance as $T \to \infty$ define the **information wall**. Pairs that separate tell us the observable that distinguishes them:
- **fBm vs CTRW**: separated cleanly via immobile fraction and increment autocorrelation.
- **SBM vs Brownian**: separates as $T$ grows via increment non-stationarity ($D(t) \sim t^{\alpha-1}$).
- **DDM vs Brownian**: separated at short times via the non-Gaussian parameter $a_2$, but asymptotically challenging once $t \gg \tau$.

In [ ]:
fig_pairwise = plot_pairwise_error_vs_length(
    lengths=clf_summary["lengths"],
    diagnostic_curves=clf_summary["diagnostic_pairwise_curves"],
)
plt.show()

## 4. Feature Ablation: Where the Information Lives

We evaluate classification performance with each feature and feature group removed (leave-one-out) and with each feature/group alone (single-feature).

In [ ]:
ablation_summary = run_step4_ablation_study(n_steps=2048, n_samples_per_class=120)
fig_ablation = plot_feature_importance_ranking(ablation_summary)
plt.show()

print("Top discriminating observables for key mechanism pairs:")
for pair_name, ranking in ablation_summary["pairwise_rankings"].items():
    top_feat = ranking[0]
    print(f"  {pair_name:20s}: {top_feat['feature']} (Accuracy: {top_feat['accuracy']:.3f})")

## 5. The Information Floor: Whittle MLE vs Cramér–Rao Bound

For Gaussian processes like fBm, the exact spectral density is known. The Whittle likelihood gives an asymptotically efficient estimator $\hat{H}$, and the Fisher information gives the Cramér–Rao lower bound (CRLB) $\mathrm{Var}(\hat{H}) \ge I(H)^{-1}$.

Comparing the Phase 1 TA-MSD least-squares estimator against the bound reveals that its variance is suboptimal and inefficient relative to the theoretical limit.

In [ ]:
info_summary = run_step5_information_floor_study(lengths=(128, 512, 2048, 8192), hurst_values=(0.3, 0.7), n_trajectories=80)
fig_info = plot_information_floor_efficiency(info_summary)
plt.show()

## 6. Closing the Loop: Conditional Exponent Estimation

Given the predicted mechanism from the classifier, we apply the mechanism-specific correction and re-estimate the anomalous exponent $\alpha$.

We compare exponent error across three conditions:
1. **Naive**: Mechanism unknown (Phase 1 TA-MSD estimator).
2. **Predicted**: Mechanism identified by classifier + conditional correction.
3. **Oracle**: Ground-truth mechanism known exactly.

The reduction from Condition 1 to Condition 2 is what Phase 2 achieved; the gap between Condition 2 and Condition 3 is the cost of misclassification.

In [ ]:
cond_summary = run_step6_conditional_exponent_study(n_steps=2048, n_samples_per_class=120)
fig_cond = plot_conditional_exponent_comparison(cond_summary)
plt.show()

print(f"Overall Exponent MAE (T = {cond_summary['n_steps']}):")
print(f"  1. Naive (Phase 1 TA-MSD)    : {cond_summary['mae_naive']:.4f}")
print(f"  2. Predicted + Correction    : {cond_summary['mae_predicted']:.4f}")
print(f"  3. Oracle (Known Mechanism)  : {cond_summary['mae_oracle']:.4f}")
print(f"Phase 2 Error Reduction: {100.0 * (cond_summary['mae_naive'] - cond_summary['mae_predicted']) / cond_summary['mae_naive']:.1f} %")